In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lesson 6: Multi-GPU Training with JAX

## Overview

In Lesson 4 you trained a small MLP on one GPU. That single GPU handled the forward pass, gradient computation, and parameter updates. With multiple GPUs, there are several ways to split the work, including data parallelism, tensor/model parallelism, and pipeline parallelism.

This lesson focuses on **data-parallel training**, the simplest starting point: each GPU receives a different slice of the batch, runs the same model and training step, and contributes gradients to a shared update. The key idea is that the training step code mostly stays the same. You change how arrays are placed on devices, and JAX handles the distributed execution.

**What you'll do:**

* Check that JAX sees multiple GPUs with `jax.devices()`
* Create a `Mesh` — a logical grid of devices with named axes
* Shard training batches across GPUs with `NamedSharding` and `PartitionSpec`
* Visualize which GPU holds which slice with `jax.debug.visualize_array_sharding`
* Run the same `jax.jit` training step on sharded data — JAX auto-parallelizes
* Rewrite the gradient computation with `shard_map` for explicit per-shard control
* Measure single-GPU vs multi-GPU throughput
* Sweep batch sizes to see how data parallelism scales

## How data-parallel training works

Data parallelism is the simplest multi-GPU strategy: split the batch, replicate the model.

1. **Replicate** the model parameters — every GPU holds a full copy of the weights.
2. **Shard** the data batch along the batch dimension — each GPU gets a different slice.
3. **Forward + backward** on each GPU independently — each computes gradients on its local slice.
4. **All-reduce** the gradients — average across GPUs so every copy sees the same update.
5. **Update** parameters identically on every GPU — same gradients means same new weights.

When each GPU has enough local compute, data parallelism can process `per_device_batch * num_gpus` examples with only a modest increase in step time compared with one GPU processing `per_device_batch`.

Data parallelism is simple and effective because every GPU runs the same model on a different slice of the batch, so the training code changes very little and throughput can improve almost linearly when the model fits comfortably on each device. The tradeoff is that each GPU must store a full copy of the model, so data parallelism does not help when the model itself is too large for one GPU; it also requires gradient synchronization across devices, which can become a bottleneck for very large models, small batches, or slower interconnects.


## Requirements

The fixed NGC image and pinned workshop requirements provide:

- `jax`, `jaxlib` — core JAX with multi-GPU support
- `optax` — optimizers (same as Lesson 4)
- `matplotlib` — throughput charts

This lesson requires **at least 2 GPUs**. The setup cell checks this and stops if only one GPU is visible.

## Setup

Import JAX and verify that multiple GPUs are visible. The sharding primitives — `Mesh`, `PartitionSpec`, and `NamedSharding` — all come from `jax.sharding`.

In [ ]:
import os

os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

import gc
import gzip
import hashlib
import html
import math
import pathlib
import shutil
import struct
import subprocess
import time
import urllib.request
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*ml_dtypes.*")
warnings.filterwarnings("ignore", message=".*JAX_PLATFORMS.*")

import jax
import jax.numpy as jnp
import optax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]
NUM_DEVICES = len(gpu_devices)

print(f"JAX version:     {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"GPU devices:     {gpu_devices}")
print(f"GPU count:       {NUM_DEVICES}")

assert len(gpu_devices) >= 2, (
    f"This lesson needs at least 2 GPUs. Found {len(gpu_devices)}. "
    f"Available devices: {devices}"
)


def block_tree(tree):
    """Wait until a PyTree of JAX arrays is ready on device."""
    return jax.block_until_ready(tree)


def drop_device_refs(*names, clear_compilation_cache=False):
    """Drop global references that may hold device buffers, then run cleanup."""
    for name in names:
        globals().pop(name, None)
    gc.collect()
    if clear_compilation_cache and hasattr(jax, "clear_caches"):
        jax.clear_caches()


def show_table(headers, rows, title=None, aligns=None):
    """Render rows as an HTML table."""
    aligns = aligns or ["left"] * len(headers)
    parts = ["<div style='font-family: system-ui; max-width: 980px;'>"]
    if title:
        parts.append(f"<h4 style='margin: 0 0 8px 0;'>{html.escape(title)}</h4>")
    parts.append("<table style='border-collapse: collapse; width: 100%; font-size: 13px;'>")
    parts.append("<thead><tr>")
    for h, a in zip(headers, aligns):
        parts.append(
            f"<th style='text-align:{a}; border-bottom:1px solid #d0d7de; padding:6px;'>"
            f"{html.escape(str(h))}</th>"
        )
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for cell, a in zip(row, aligns):
            parts.append(
                f"<td style='text-align:{a}; border-bottom:1px solid #eef1f4; padding:6px;'>"
                f"{html.escape(str(cell))}</td>"
            )
        parts.append("</tr>")
    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))


def show_bars(rows, title, unit="", lower_is_better=False):
    """Render (label, value) pairs as a horizontal bar chart in HTML."""
    max_value = max(float(value) for _, value in rows) or 1.0
    color = "#1a7f37" if not lower_is_better else "#0969da"
    parts = ["<div style='font-family: Arial, sans-serif; max-width: 760px;'>"]
    parts.append(f"<h4 style='margin: 0 0 8px 0;'>{html.escape(title)}</h4>")
    for label, value in rows:
        width = max(3, 100 * float(value) / max_value)
        parts.append(
            "<div style='display:grid; grid-template-columns: 190px 1fr 130px; gap: 8px; "
            "align-items:center; margin: 6px 0;'>"
            f"<div style='font-size:13px;'>{html.escape(str(label))}</div>"
            "<div style='background:#f6f8fa; border-radius:6px; overflow:hidden; height:22px;'>"
            f"<div style='height:22px; width:{width:.1f}%; background:{color};'></div></div>"
            f"<div style='font-size:13px; font-variant-numeric: tabular-nums;'>{float(value):,.1f} {html.escape(unit)}</div>"
            "</div>"
        )
    parts.append(
        f"<div style='font-size:12px; color:#57606a;'>"
        f"{'Lower' if lower_is_better else 'Higher'} is better.</div></div>"
    )
    display(HTML("".join(parts)))

## Load Fashion-MNIST and define the MLP

This uses the same Fashion-MNIST dataset from Lesson 4, but with a deliberately compute-heavy shared-block MLP so the multi-GPU benefit is easier to see. If you ran Lesson 4, the data files are cached locally.

The benchmark uses Fashion-MNIST images and labels. Dataset loading and host-side batch preparation happen before timing; the benchmark itself measures the compiled GPU training step.

The model applies one shared hidden block many times. Reusing the same block increases local compute without increasing the number of gradient values that must be synchronized across GPUs.


In [ ]:
DATA_DIR = pathlib.Path.home() / ".cache" / "jax-course" / "fashion-mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
}
PRIMARY_BASE_URL = "https://github.com/zalandoresearch/fashion-mnist/raw/master/data/fashion"
FALLBACK_BASE_URL = "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com"


def md5sum(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_if_needed(filename, expected_md5):
    path = DATA_DIR / filename
    if path.exists() and md5sum(path) == expected_md5:
        return path
    for base in [PRIMARY_BASE_URL, FALLBACK_BASE_URL]:
        try:
            print(f"Downloading {filename}")
            urllib.request.urlretrieve(f"{base}/{filename}", path)
            if md5sum(path) != expected_md5:
                raise ValueError("MD5 mismatch")
            return path
        except (OSError, ValueError):
            if path.exists():
                path.unlink()
    raise RuntimeError(f"Could not download {filename}")


def read_idx_images(path):
    with gzip.open(path, "rb") as f:
        _, n, rows, cols = struct.unpack(">IIII", f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols)


def read_idx_labels(path):
    with gzip.open(path, "rb") as f:
        _, n = struct.unpack(">II", f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n)


paths = {name: download_if_needed(name, cs) for name, cs in FILES.items()}
train_images = read_idx_images(paths["train-images-idx3-ubyte.gz"])
train_labels = read_idx_labels(paths["train-labels-idx1-ubyte.gz"])

# Shuffle once on the host so each benchmark batch is a realistic class mix.
perm = np.random.default_rng(0).permutation(len(train_images))
x_train_all = (train_images[perm].astype(np.float32) / 255.0).reshape(len(train_images), -1)
y_train_all = train_labels[perm].astype(np.int32)

drop_device_refs("train_images", "train_labels", "perm")


INPUT_DIM = 28 * 28
WIDTH = 1024
NUM_CLASSES = 10
BLOCK_REPEATS = 128
BLOCK_MIX = 0.10
LEARNING_RATE = 3e-4

PER_DEVICE_BATCH = 1024
GLOBAL_BATCH = PER_DEVICE_BATCH * NUM_DEVICES
NUM_TRAIN_BATCHES = 8
BENCHMARK_WARMUP = 4
BENCHMARK_STEPS = 15
BENCHMARK_REPEATS = 3


def init_params(seed=0):
    rng = np.random.default_rng(seed)

    def normal(shape, scale):
        return rng.standard_normal(shape).astype(np.float32) * scale

    return {
        "w_in": normal((INPUT_DIM, WIDTH), math.sqrt(2.0 / INPUT_DIM)),
        "b_in": np.zeros((WIDTH,), dtype=np.float32),
        "w_block": normal((WIDTH, WIDTH), math.sqrt(2.0 / WIDTH)),
        "b_block": np.zeros((WIDTH,), dtype=np.float32),
        "w_out": normal((WIDTH, NUM_CLASSES), math.sqrt(2.0 / WIDTH)),
        "b_out": np.zeros((NUM_CLASSES,), dtype=np.float32),
    }


def make_fashion_batches(batch_size, num_batches=NUM_TRAIN_BATCHES):
    available_batches = len(x_train_all) // batch_size
    if available_batches == 0:
        raise ValueError(
            f"Batch size {batch_size:,} exceeds the "
            f"{len(x_train_all):,}-example dataset."
        )

    # Use as many complete batches as the dataset can provide.
    num_batches = min(num_batches, available_batches)
    needed = batch_size * num_batches
    x = x_train_all[:needed].reshape(num_batches, batch_size, INPUT_DIM)
    y = y_train_all[:needed].reshape(num_batches, batch_size)
    return x, y


def model(params, x):
    h = jax.nn.gelu(x @ params["w_in"] + params["b_in"])

    def block(h, _):
        z = jax.nn.gelu(h @ params["w_block"] + params["b_block"])
        h = (1.0 - BLOCK_MIX) * h + BLOCK_MIX * z
        return h, None

    h, _ = jax.lax.scan(block, h, xs=None, length=BLOCK_REPEATS)
    return h @ params["w_out"] + params["b_out"]


def loss_with_metrics(params, batch):
    x, y = batch
    logits = model(params, x)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
    accuracy = jnp.mean(jnp.argmax(logits, axis=-1) == y)
    return loss, {"accuracy": accuracy}


optimizer = optax.adamw(learning_rate=LEARNING_RATE, weight_decay=1e-4)
param_template = init_params(seed=1)
PARAM_COUNT = sum(x.size for x in param_template.values())
GRADIENT_MB = PARAM_COUNT * np.dtype(np.float32).itemsize / 1e6

drop_device_refs("param_template")

show_table(
    ["", "Value"],
    [
        ("Dataset", f"Fashion-MNIST train ({len(x_train_all):,} examples)"),
        ("Input shape", "28 x 28 grayscale, flattened to 784"),
        ("Model", f"shared-block MLP, width={WIDTH}, repeats={BLOCK_REPEATS}"),
        ("Parameters", f"{PARAM_COUNT:,}"),
        ("Gradient size", f"{GRADIENT_MB:.1f} MB per step"),
        ("Per-GPU batch", PER_DEVICE_BATCH),
        ("Global batch on all GPUs", GLOBAL_BATCH),
        ("Benchmark", f"median of {BENCHMARK_REPEATS} x {BENCHMARK_STEPS} steps"),
    ],
    title="Fashion-MNIST compute-heavy workload",
)

## Single-GPU baseline

The baseline uses one GPU with batch size `PER_DEVICE_BATCH`. This is the amount of work each GPU will get in the multi-GPU run.

In [ ]:
single_device = gpu_devices[0]

x_batches_1gpu, y_batches_1gpu = make_fashion_batches(PER_DEVICE_BATCH)
x_batches_1gpu = jax.device_put(x_batches_1gpu, single_device)
y_batches_1gpu = jax.device_put(y_batches_1gpu, single_device)

params_1gpu = jax.device_put(init_params(seed=1), single_device)
opt_state_1gpu = optimizer.init(params_1gpu)


@jax.jit
def train_step(params, opt_state, batch):
    (loss, metrics), grads = jax.value_and_grad(loss_with_metrics, has_aux=True)(
        params,
        batch,
    )
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, {"loss": loss, "accuracy": metrics["accuracy"]}


def benchmark_training(
    step_fn,
    params,
    opt_state,
    x_batches,
    y_batches,
    warmup=BENCHMARK_WARMUP,
    steps=BENCHMARK_STEPS,
    repeats=BENCHMARK_REPEATS,
):
    """Warm up, then report median steady-state throughput."""
    num_batches = x_batches.shape[0]
    for i in range(warmup):
        batch = (x_batches[i % num_batches], y_batches[i % num_batches])
        params, opt_state, _ = step_fn(params, opt_state, batch)
    block_tree((params, opt_state))

    batch_size = x_batches.shape[1]
    timings = []
    metrics = None
    for repeat in range(repeats):
        start = time.perf_counter()
        for i in range(steps):
            batch_index = (repeat * steps + i) % num_batches
            batch = (x_batches[batch_index], y_batches[batch_index])
            params, opt_state, metrics = step_fn(params, opt_state, batch)
        params, opt_state, metrics = block_tree((params, opt_state, metrics))
        timings.append(time.perf_counter() - start)

    elapsed = float(np.median(timings))
    return {
        "examples_per_sec": steps * batch_size / elapsed,
        "ms_per_step": 1000 * elapsed / steps,
        "final_loss": float(metrics["loss"]),
        "final_accuracy": float(metrics["accuracy"]),
    }


result_1gpu = benchmark_training(
    train_step,
    params_1gpu,
    opt_state_1gpu,
    x_batches_1gpu,
    y_batches_1gpu,
)

show_table(
    ["Metric", "Value"],
    [
        ("GPUs used", "1"),
        ("Batch per step", PER_DEVICE_BATCH),
        ("Throughput", f"{result_1gpu['examples_per_sec']:,.0f} examples/sec"),
        ("Step time", f"{result_1gpu['ms_per_step']:.2f} ms"),
        ("Final loss", f"{result_1gpu['final_loss']:.4f}"),
        ("Final accuracy", f"{100 * result_1gpu['final_accuracy']:.1f}%"),
    ],
    title="Single-GPU baseline",
)

# Keep scalar timing results, but free device buffers from the single-GPU run.
drop_device_refs(
    "params_1gpu",
    "opt_state_1gpu",
    "x_batches_1gpu",
    "y_batches_1gpu",
)

## The device mesh

A `Mesh` maps physical GPUs to a logical grid with named axes. For data parallelism, we create a 1-D mesh with all GPUs along a single `'data'` axis.

The table below summarizes the small set of sharding primitives we will use to describe that placement explicitly.

| Primitive | Purpose |
| --- | --- |
| `Mesh(devices, axis_names)` | Logical grid of GPUs |
| `PartitionSpec('data', None)` | Shard dim 0 along `'data'`, replicate dim 1 |
| `NamedSharding(mesh, spec)` | Combines mesh + partition spec into a placement plan |
| `jax.device_put(x, sharding)` | Places array on devices according to the sharding |


> **Axis names are user-defined.** We call the mesh axis `'data'` because this lesson shards the batch for data parallelism. In larger models you might use names like `'model'`, `'tensor'`, `'pipeline'`, or `'fsdp'` to describe other kinds of parallelism. For example, a 2-D mesh might use `('data', 'model')`, where one axis shards the batch and the other shards model weights or activations. The names do not have special meaning to JAX; they become meaningful through the `PartitionSpec`s and collectives that refer to them.

In practice, these pieces are used together: `PartitionSpec` describes the layout, `NamedSharding` attaches that layout to a mesh of devices, and `jax.device_put` moves an array into that layout.

Let's define the mesh now.

In [ ]:
mesh = Mesh(np.array(gpu_devices), ("data",))

show_table(
    ["", "Value"],
    [
        ("Mesh shape", str(mesh.shape)),
        ("Axis names", str(mesh.axis_names)),
        ("Devices", ", ".join(str(d) for d in mesh.devices.flat)),
    ],
    title="Device mesh",
)

## GPU topology check

Data-parallel training synchronizes gradients every step, so inter-GPU bandwidth matters. `NVLink`/`NV*` paths are usually much better for this benchmark than `PHB` paths through host/PCIe.

In [ ]:
if shutil.which("nvidia-smi"):
    topo = subprocess.run(
        ["nvidia-smi", "topo", "-m"],
        check=False,
        text=True,
        capture_output=True,
    )
    print(topo.stdout or topo.stderr)
else:
    print("nvidia-smi is not available in this environment.")

## Shard data, replicate parameters

In data-parallel training:
- **Data** is sharded along the batch dimension — each GPU gets a different slice of the batch
- **Parameters** are replicated — every GPU holds a full copy so the forward pass runs identically

`PartitionSpec('data', None)` means: split the first dimension across the `'data'` mesh axis, replicate the second dimension. `PartitionSpec()` with no arguments means: replicate everything.

After calling `jax.device_put`, use `jax.debug.visualize_array_sharding` to see the layout — it prints a text grid showing which GPU holds which portion of the array.

In [ ]:
batch_data_sharding = NamedSharding(mesh, P("data", None))
batch_label_sharding = NamedSharding(mesh, P("data"))
all_data_sharding = NamedSharding(mesh, P(None, "data", None))
all_label_sharding = NamedSharding(mesh, P(None, "data"))
replicated = NamedSharding(mesh, P())

x_batches_multi, y_batches_multi = make_fashion_batches(GLOBAL_BATCH)
x_batches_multi = jax.device_put(x_batches_multi, all_data_sharding)
y_batches_multi = jax.device_put(y_batches_multi, all_label_sharding)

params_multi = jax.device_put(init_params(seed=1), replicated)
opt_state_multi = optimizer.init(params_multi)

print(
    f"Global batch: {GLOBAL_BATCH} examples "
    f"({PER_DEVICE_BATCH} per GPU x {NUM_DEVICES} GPUs)"
)
print(f"Training batches shape: {x_batches_multi.shape}")
print()

print("One training batch: sharded along the batch dimension")
jax.debug.visualize_array_sharding(x_batches_multi[0])

print()
print("Weight w_block: replicated on all GPUs")
jax.debug.visualize_array_sharding(params_multi["w_block"])

## Data-parallel training — the automatic way

The training step code does not change. The same `jax.jit`-compiled `train_step` from the single-GPU baseline works on sharded inputs. When JAX sees that the batch is sharded across GPUs and the parameters are replicated, it automatically:

1. Runs the forward pass on each GPU's data slice
2. Computes per-shard gradients
3. Inserts an all-reduce to average gradients across GPUs
4. Updates parameters identically on every GPU

You do not need to write any communication code. The parallelism comes entirely from how the arrays are placed.

In [ ]:
result_multi = benchmark_training(
    train_step,
    params_multi,
    opt_state_multi,
    x_batches_multi,
    y_batches_multi,
)

show_table(
    ["Metric", "Value"],
    [
        ("GPUs used", NUM_DEVICES),
        ("Global batch", GLOBAL_BATCH),
        ("Per-GPU batch", PER_DEVICE_BATCH),
        ("Throughput", f"{result_multi['examples_per_sec']:,.0f} examples/sec"),
        ("Step time", f"{result_multi['ms_per_step']:.2f} ms"),
        ("Final loss", f"{result_multi['final_loss']:.4f}"),
        ("Final accuracy", f"{100 * result_multi['final_accuracy']:.1f}%"),
    ],
    title=f"Data-parallel training on {NUM_DEVICES} GPUs",
)

## Single-GPU vs multi-GPU throughput

This is the headline data-parallel result. The per-GPU batch is the same in both runs; the multi-GPU run processes more examples per step because each GPU receives its own shard.

Here, **faster** means higher training throughput in examples per second.

In [ ]:
speed_ratio = result_multi["examples_per_sec"] / result_1gpu["examples_per_sec"]

show_table(
    ["", "1 GPU", f"{NUM_DEVICES} GPUs", "Throughput ratio"],
    [
        ("Per-GPU batch", PER_DEVICE_BATCH, PER_DEVICE_BATCH, "same"),
        ("Global batch", PER_DEVICE_BATCH, GLOBAL_BATCH, f"{NUM_DEVICES}x"),
        (
            "Examples/sec",
            f"{result_1gpu['examples_per_sec']:,.0f}",
            f"{result_multi['examples_per_sec']:,.0f}",
            f"{speed_ratio:.2f}x",
        ),
        (
            "ms/step",
            f"{result_1gpu['ms_per_step']:.2f}",
            f"{result_multi['ms_per_step']:.2f}",
            "",
        ),
    ],
    title="Weak-scaling throughput: same per-GPU batch",
    aligns=["left", "right", "right", "right"],
)

show_bars(
    [
        ("1 GPU", result_1gpu["examples_per_sec"]),
        (f"{NUM_DEVICES} GPUs", result_multi["examples_per_sec"]),
    ],
    "Training throughput (examples/sec)",
    "examples/s",
)

step_ratio = result_multi["ms_per_step"] / result_1gpu["ms_per_step"]
if speed_ratio >= 1.0:
    message = (
        f"The multi-GPU run is faster for this Fashion-MNIST workload: "
        f"throughput improves by {speed_ratio:.2f}x. Each GPU still processes "
        f"{PER_DEVICE_BATCH} examples, while the global batch increases from "
        f"{PER_DEVICE_BATCH} to {GLOBAL_BATCH}. Step time changes by {step_ratio:.2f}x, "
        f"so the larger batch translates into higher examples/sec."
    )
else:
    message = (
        f"This run is still communication-bound: throughput changes by {speed_ratio:.2f}x. "
        f"Increase BLOCK_REPEATS or PER_DEVICE_BATCH to give each GPU more local work."
    )

border_color = "#1a7f37" if speed_ratio >= 1.0 else "#d1242f"
display(HTML(
    "<div style='font-family: system-ui; max-width: 900px; "
    f"border-left: 4px solid {border_color}; padding: 10px 12px; "
    "background: #f6f8fa; margin: 12px 0;'>"
    f"{html.escape(message)}"
    "</div>"
))

## Explicit data parallelism with shard_map

The automatic approach covers most data-parallel workloads. But sometimes you want to see — or control — exactly what each GPU computes. `shard_map` lets you write a function that operates on **per-shard** arrays and use explicit collectives to communicate across devices.

Inside a `shard_map` function:
- Each GPU receives its **local shard** — for example, `(256, 784)` instead of `(512, 784)`
- `in_specs` declares how inputs are sliced: `P('data', None)` shards dim 0 across GPUs
- `out_specs` declares how outputs are reassembled: `P()` means the output is the same on all GPUs
- `jax.lax.pmean(x, 'data')` averages `x` across all GPUs along the `'data'` axis

In [ ]:
@partial(
    jax.shard_map,
    mesh=mesh,
    in_specs=(P(), P("data", None), P("data",)),
    out_specs=(P(), P(), P()),
)
def compute_grads_shardmap(params, x_shard, y_shard):
    (loss, metrics), grads = jax.value_and_grad(loss_with_metrics, has_aux=True)(
        params,
        (x_shard, y_shard),
    )
    grads = jax.lax.pmean(grads, "data")
    loss = jax.lax.pmean(loss, "data")
    accuracy = jax.lax.pmean(metrics["accuracy"], "data")
    return grads, loss, accuracy


@jax.jit
def train_step_explicit(params, opt_state, batch):
    x, y = batch
    grads, loss, accuracy = compute_grads_shardmap(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, {"loss": loss, "accuracy": accuracy}


params_explicit = jax.device_put(init_params(seed=1), replicated)
opt_state_explicit = optimizer.init(params_explicit)

result_explicit = benchmark_training(
    train_step_explicit,
    params_explicit,
    opt_state_explicit,
    x_batches_multi,
    y_batches_multi,
)

show_table(
    ["Approach", "Examples/sec", "ms/step"],
    [
        (
            "jit on sharded arrays",
            f"{result_multi['examples_per_sec']:,.0f}",
            f"{result_multi['ms_per_step']:.2f}",
        ),
        (
            "shard_map explicit",
            f"{result_explicit['examples_per_sec']:,.0f}",
            f"{result_explicit['ms_per_step']:.2f}",
        ),
    ],
    title="Automatic vs explicit data parallelism",
    aligns=["left", "right", "right"],
)

drop_device_refs(
    "params_multi",
    "opt_state_multi",
    "params_explicit",
    "opt_state_explicit",
    "x_batches_multi",
    "y_batches_multi",
)

## When to use which

| Approach | How it works | When to use |
| --- | --- | --- |
| `jit` on sharded arrays | JAX infers parallelism from input sharding | Default for data parallelism — covers most cases |
| `shard_map` | You write per-shard code with explicit collectives | When you need fine control: custom reductions, pipeline parallelism, debugging |

For standard data-parallel training, the automatic approach is simpler and usually the right default. Use `shard_map` when you need explicit per-shard control or want to customize what happens on each device.


## Batch-size scaling

Data parallelism lets you scale the global batch size with the number of GPUs. Larger batches amortize kernel launch overhead and improve GPU utilization — up to the point where per-device memory or communication becomes the bottleneck.

The sweep below tests different global batch sizes across all GPUs.

In [ ]:
BATCH_SIZES = [256, 512, 1024, 2048, 4096]

scaling_results = []

for bs in BATCH_SIZES:
    if bs % NUM_DEVICES != 0:
        print(f"Skipping global batch {bs}: not divisible by {NUM_DEVICES} GPUs.")
        continue

    try:
        x_bs, y_bs = make_fashion_batches(bs)

        x_bs = jax.device_put(x_bs, all_data_sharding)
        y_bs = jax.device_put(y_bs, all_label_sharding)

        params_bs = jax.device_put(init_params(seed=1), replicated)
        opt_bs = optimizer.init(params_bs)

        result = benchmark_training(
            train_step,
            params_bs,
            opt_bs,
            x_bs,
            y_bs,
        )

        scaling_results.append(
            {
                "batch_size": bs,
                "per_device": bs // NUM_DEVICES,
                "examples_per_sec": result["examples_per_sec"],
                "ms_per_step": result["ms_per_step"],
            }
        )

    except Exception as e:  # noqa: BLE001 - resource failures vary by GPU and backend
        print(f"Batch size {bs}: {e}")

    finally:
        drop_device_refs("x_bs", "y_bs", "params_bs", "opt_bs", "result")


show_table(
    ["Global batch", "Per GPU", "Examples/sec", "ms/step"],
    [
        (
            r["batch_size"],
            r["per_device"],
            f"{r['examples_per_sec']:,.0f}",
            f"{r['ms_per_step']:.2f}",
        )
        for r in scaling_results
    ],
    title=f"Batch-size scaling on {NUM_DEVICES} GPUs",
    aligns=["right", "right", "right", "right"],
)


fig, ax = plt.subplots(figsize=(8, 5))

batches = [r["batch_size"] for r in scaling_results]
throughputs = [r["examples_per_sec"] for r in scaling_results]

ax.plot(
    batches,
    throughputs,
    "o-",
    color="#0969da",
    linewidth=2,
    markersize=8,
)

ax.set_xlabel("Global batch size")
ax.set_ylabel("Examples per second")
ax.set_title(f"Throughput vs batch size — {NUM_DEVICES} GPUs data-parallel")
ax.set_xscale("log", base=2)
ax.set_xticks(batches)
ax.set_xticklabels([str(b) for b in batches])
ax.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## Summary

This lesson moved from one GPU to many by changing how arrays are placed on devices — the training step code stayed the same.

* **`Mesh(devices, axis_names)`** maps physical GPUs to a logical grid with named axes.
* **`PartitionSpec`** declares how array dimensions map to mesh axes. `P('data', None)` shards the batch dimension, replicates the features.
* **`NamedSharding(mesh, spec)`** combines mesh + spec into a placement plan for `jax.device_put`.
* **`jax.debug.visualize_array_sharding`** shows which GPU holds which slice — use it after every placement change.
* **Automatic parallelism**: `jax.jit` on sharded inputs inserts all-reduce and per-shard computation automatically. No code changes needed.
* **`shard_map`** gives explicit per-shard control with `jax.lax.pmean` for gradient averaging — useful when you need to customize the communication pattern.
* **Batch-size scaling**: larger global batches can improve throughput until GPU utilization, memory, or communication becomes the bottleneck.

In the next lesson, you will build a full transformer model that combines the attention mechanism from Lesson 5 with the multi-GPU training from this lesson.

Official references:

* [Distributed arrays and automatic parallelization](https://docs.jax.dev/en/latest/notebooks/Distributed_arrays_and_automatic_parallelization.html)
* [Introduction to shard_map](https://docs.jax.dev/en/latest/notebooks/shard_map.html)
* [jax.debug.visualize_array_sharding](https://docs.jax.dev/en/latest/_autosummary/jax.debug.visualize_array_sharding.html)
